Import + load đúng dataset G06

In [4]:
# ===== G09 - CELL 1: SETUP & LOAD DATA =====

import pandas as pd
import numpy as np
import warnings

from statsmodels.tsa.holtwinters import (
    ExponentialSmoothing,
    Holt,
    SimpleExpSmoothing
)

warnings.filterwarnings("ignore")

# Load chính dataset đã khóa từ G06
df = pd.read_csv("../output/G06_forecast_dataset.csv")

df["Month"] = pd.to_datetime(df["Month"])

print("=== G09 DATA CHECK ===")
print("Rows:", len(df))
print("Materials:", df["MaterialKey"].nunique())
print("Min month:", df["Month"].min())
print("Max month:", df["Month"].max())

print("\nTrain rows:",
      (df["IsTrainPeriod"] == 1).sum())

print("Test rows:",
      (df["IsTestPeriod"] == 1).sum())

print("\nTier distribution:")
print(
    df[["MaterialKey", "DSSEligibilityTier"]]
    .drop_duplicates()["DSSEligibilityTier"]
    .value_counts()
)

=== G09 DATA CHECK ===
Rows: 1145
Materials: 35
Min month: 2022-07-01 00:00:00
Max month: 2025-05-01 00:00:00

Train rows: 941
Test rows: 204

Tier distribution:
DSSEligibilityTier
A    28
B     7
Name: count, dtype: int64


kiểm tra lịch sử train

In [5]:
# ===== G09 - CELL 2: CHECK ETS DATA ELIGIBILITY =====

ets_eligibility = (
    df.groupby(
        ["MaterialKey", "ProductNumber",
         "MaterialName", "DSSEligibilityTier"]
    )
    .agg(
        TrainObserved=(
            "IsTrainPeriod",
            lambda x: (x == 1).sum()
        ),
        TestObserved=(
            "IsTestPeriod",
            lambda x: (x == 1).sum()
        )
    )
    .reset_index()
)

# Seasonal Holt-Winters với seasonal_periods=12
# nên có ít nhất 2 chu kỳ đầy đủ = 24 observations
ets_eligibility["Has24TrainObs"] = (
    ets_eligibility["TrainObserved"] >= 24
)

ets_eligibility["HasFullTest6"] = (
    ets_eligibility["TestObserved"] == 6
)

print("=== ETS ELIGIBILITY CHECK ===")

print("\nTrain observation counts:")
print(
    ets_eligibility["TrainObserved"]
    .value_counts()
    .sort_index()
)

print("\nMaterials with >= 24 train observations:",
      ets_eligibility["Has24TrainObs"].sum())

print("Materials with < 24 train observations:",
      (~ets_eligibility["Has24TrainObs"]).sum())

print("Materials with full 6-month test:",
      ets_eligibility["HasFullTest6"].sum())

print("\n=== MATERIALS WITH < 24 TRAIN OBSERVATIONS ===")

display(
    ets_eligibility[
        ~ets_eligibility["Has24TrainObs"]
    ]
)

=== ETS ELIGIBILITY CHECK ===

Train observation counts:
TrainObserved
6      2
18     2
23     1
29    30
Name: count, dtype: int64

Materials with >= 24 train observations: 30
Materials with < 24 train observations: 5
Materials with full 6-month test: 34

=== MATERIALS WITH < 24 TRAIN OBSERVATIONS ===


,MaterialKey,ProductNumber,MaterialName,DSSEligibilityTier,TrainObserved,TestObserved,Has24TrainObs,HasFullTest6
0,4,CA-5965,LL Crankarm,B,6,6,False,True
20,167,RM-M464,LL Mountain Rim,B,18,6,False,True
21,168,RM-M692,ML Mountain Rim,B,18,6,False,True
22,169,RM-M823,HL Mountain Rim,B,23,0,False,False
26,173,RM-T801,Touring Rim,B,6,6,False,True


khai báo metric + hàm fit ETS an toàn

In [6]:
# ===== G09 - CELL 3: METRICS & SAFE ETS FITTER =====

def mae(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return np.mean(np.abs(forecast - actual))


def rmse(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return np.sqrt(np.mean((forecast - actual) ** 2))


def bias(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return np.mean(forecast - actual)


def fit_ets_model(train_values, model_name, horizon=6):
    """
    Fit one ETS/Holt-Winters candidate and return forecasts.

    Important:
    - Uses TRAIN data only.
    - No test actual values are used during forecasting.
    - Seasonal ETS requires >= 24 training observations.
    """

    y = np.asarray(train_values, dtype=float)

    if model_name == "ETS_Level":

        model = SimpleExpSmoothing(
            y,
            initialization_method="estimated"
        )

    elif model_name == "ETS_Trend":

        model = Holt(
            y,
            initialization_method="estimated"
        )

    elif model_name == "ETS_Seasonal":

        if len(y) < 24:
            raise ValueError(
                f"Seasonal ETS requires >=24 train observations; got {len(y)}"
            )

        model = ExponentialSmoothing(
            y,
            trend="add",
            seasonal="add",
            seasonal_periods=12,
            initialization_method="estimated"
        )

    else:
        raise ValueError(
            f"Unknown ETS model: {model_name}"
        )

    fitted = model.fit(optimized=True)

    forecast = np.asarray(
        fitted.forecast(horizon),
        dtype=float
    )

    return forecast


# ===== QUICK FUNCTION TEST =====

sample_material = (
    ets_eligibility[
        ets_eligibility["Has24TrainObs"]
    ]["MaterialKey"]
    .iloc[0]
)

sample_train = (
    df[
        (df["MaterialKey"] == sample_material)
        & (df["IsTrainPeriod"] == 1)
    ]
    .sort_values("Month")["RequirementQty_PU"]
    .to_numpy()
)

print("Sample MaterialKey:", sample_material)
print("Train observations:", len(sample_train))

for model_name in [
    "ETS_Level",
    "ETS_Trend",
    "ETS_Seasonal"
]:
    try:
        forecast = fit_ets_model(
            sample_train,
            model_name,
            horizon=6
        )

        print(
            model_name,
            "PASS",
            "Forecast:",
            np.round(forecast, 2)
        )

    except Exception as e:
        print(
            model_name,
            "FAIL",
            type(e).__name__,
            str(e)
        )

Sample MaterialKey: 5
Train observations: 29
ETS_Level PASS Forecast: [4163.29 4163.29 4163.29 4163.29 4163.29 4163.29]
ETS_Trend PASS Forecast: [4769.79 4893.35 5016.9  5140.45 5264.   5387.56]
ETS_Seasonal PASS Forecast: [1963.07 5527.16 -633.75 1452.03 2584.8  1073.52]


chạy ETS cho toàn bộ materials

In [7]:
# ===== G09 - CELL 4: RUN ETS BACKTEST FOR ALL MATERIALS =====

ETS_MODELS = [
    "ETS_Level",
    "ETS_Trend",
    "ETS_Seasonal"
]

ets_predictions = []
ets_metrics = []
ets_logs = []

for material_key, group in df.groupby("MaterialKey"):

    group = group.sort_values("Month")

    train = group[group["IsTrainPeriod"] == 1].copy()
    test = group[group["IsTestPeriod"] == 1].copy()

    train_values = (
        train["RequirementQty_PU"]
        .astype(float)
        .to_numpy()
    )

    # -------------------------------------------------
    # Run each ETS candidate
    # -------------------------------------------------
    for model_name in ETS_MODELS:

        # Material không có đủ 6 actual test months
        if len(test) != 6:

            status = (
                f"INSUFFICIENT_TEST_DATA_{len(test)}_OF_6"
            )

            ets_metrics.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Status": status
            })

            ets_logs.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "TrainObserved": len(train),
                "TestObserved": len(test),
                "NegativeForecastsRaw": np.nan,
                "Status": status,
                "Message": ""
            })

            continue

        # Seasonal ETS cần ít nhất 24 train observations
        if (
            model_name == "ETS_Seasonal"
            and len(train) < 24
        ):

            status = (
                f"NOT_ELIGIBLE_SEASONAL_"
                f"TRAIN_{len(train)}_LT_24"
            )

            ets_metrics.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Status": status
            })

            ets_logs.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "TrainObserved": len(train),
                "TestObserved": len(test),
                "NegativeForecastsRaw": np.nan,
                "Status": status,
                "Message": ""
            })

            continue

        try:
            # -----------------------------
            # FIT + FORECAST
            # -----------------------------
            raw_forecast = fit_ets_model(
                train_values,
                model_name,
                horizon=6
            )

            # Đếm forecast âm trước khi clipping
            negative_count = int(
                (raw_forecast < 0).sum()
            )

            # Business constraint:
            # material requirement cannot be negative
            forecast_values = np.maximum(
                raw_forecast,
                0
            )

            actual_values = (
                test["RequirementQty_PU"]
                .astype(float)
                .to_numpy()
            )

            # -----------------------------
            # STORE PREDICTIONS
            # -----------------------------
            for month, actual, raw_f, forecast in zip(
                test["Month"],
                actual_values,
                raw_forecast,
                forecast_values
            ):

                ets_predictions.append({
                    "MaterialKey": material_key,
                    "ForecastMonth": month,
                    "Model": model_name,
                    "ActualQty": actual,
                    "RawForecastQty": raw_f,
                    "ForecastQty": forecast,
                    "WasClipped": int(raw_f < 0),
                    "Error": forecast - actual
                })

            # -----------------------------
            # STORE METRICS
            # -----------------------------
            ets_metrics.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "MAE": mae(
                    actual_values,
                    forecast_values
                ),
                "RMSE": rmse(
                    actual_values,
                    forecast_values
                ),
                "Bias": bias(
                    actual_values,
                    forecast_values
                ),
                "Status": "SUCCESS"
            })

            # -----------------------------
            # STORE LOG
            # -----------------------------
            ets_logs.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "TrainObserved": len(train),
                "TestObserved": len(test),
                "NegativeForecastsRaw": negative_count,
                "Status": "SUCCESS",
                "Message": (
                    "Negative forecasts clipped to 0"
                    if negative_count > 0
                    else ""
                )
            })

        except Exception as e:

            status = "FIT_FAILED"

            ets_metrics.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Status": status
            })

            ets_logs.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "TrainObserved": len(train),
                "TestObserved": len(test),
                "NegativeForecastsRaw": np.nan,
                "Status": status,
                "Message": (
                    f"{type(e).__name__}: {str(e)}"
                )
            })


# Convert to DataFrames
ets_predictions = pd.DataFrame(ets_predictions)
ets_metrics = pd.DataFrame(ets_metrics)
ets_logs = pd.DataFrame(ets_logs)


# ===== SUMMARY =====

print("=== ETS BATCH RESULTS ===")

print("\nPrediction rows:")
print(len(ets_predictions))

print("\nMetric rows:")
print(len(ets_metrics))

print("\nStatus by model:")
print(
    ets_metrics
    .groupby(["Model", "Status"])
    .size()
)

print("\nRaw negative forecasts:")
print(
    ets_logs["NegativeForecastsRaw"]
    .fillna(0)
    .sum()
)

print("\nFit failures:")
print(
    (ets_metrics["Status"] == "FIT_FAILED").sum()
)

=== ETS BATCH RESULTS ===

Prediction rows:
588

Metric rows:
105

Status by model:
Model         Status                              
ETS_Level     INSUFFICIENT_TEST_DATA_0_OF_6            1
              SUCCESS                                 34
ETS_Seasonal  INSUFFICIENT_TEST_DATA_0_OF_6            1
              NOT_ELIGIBLE_SEASONAL_TRAIN_18_LT_24     2
              NOT_ELIGIBLE_SEASONAL_TRAIN_6_LT_24      2
              SUCCESS                                 30
ETS_Trend     INSUFFICIENT_TEST_DATA_0_OF_6            1
              SUCCESS                                 34
dtype: int64

Raw negative forecasts:
3.0

Fit failures:
0


kiểm tra 3 forecast âm + chọn ETS tốt nhất

Cell 6: Export kết quả

In [ ]:
# ===== G09 - CELL 5: VALIDATE & SELECT BEST ETS =====

print("=== CLIPPED NEGATIVE FORECASTS ===")

clipped_forecasts = ets_predictions[
    ets_predictions["WasClipped"] == 1
].copy()

display(
    clipped_forecasts[
        [
            "MaterialKey",
            "ForecastMonth",
            "Model",
            "ActualQty",
            "RawForecastQty",
            "ForecastQty"
        ]
    ]
)


# ===== BASIC QA =====

print("\n=== ETS QA ===")

print("Prediction rows:", len(ets_predictions))

print(
    "Duplicate Material-Month-Model:",
    ets_predictions.duplicated(
        subset=[
            "MaterialKey",
            "ForecastMonth",
            "Model"
        ]
    ).sum()
)

print(
    "NULL Actual:",
    ets_predictions["ActualQty"].isna().sum()
)

print(
    "NULL Forecast:",
    ets_predictions["ForecastQty"].isna().sum()
)

print(
    "Negative final forecasts:",
    (ets_predictions["ForecastQty"] < 0).sum()
)


# ===== SELECT BEST ETS BY MAE =====

successful_ets = ets_metrics[
    ets_metrics["Status"] == "SUCCESS"
].copy()

best_ets = (
    successful_ets
    .sort_values(
        ["MaterialKey", "MAE", "RMSE", "Model"]
    )
    .groupby("MaterialKey")
    .first()
    .reset_index()
)


print("\n=== BEST ETS COUNT ===")

print(
    best_ets["Model"]
    .value_counts()
)


print("\n=== BEST ETS BY MATERIAL ===")

display(
    best_ets[
        [
            "MaterialKey",
            "Model",
            "MAE",
            "RMSE",
            "Bias"
        ]
    ]
)

=== CLIPPED NEGATIVE FORECASTS ===


,MaterialKey,ForecastMonth,Model,ActualQty,RawForecastQty,ForecastQty
26,5,2025-02-01,ETS_Seasonal,934.0,-633.745429,0.0
392,170,2025-02-01,ETS_Seasonal,250.0,-646.272675,0.0
428,172,2025-02-01,ETS_Seasonal,90.0,-34.678852,0.0



=== ETS QA ===
Prediction rows: 588
Duplicate Material-Month-Model: 0
NULL Actual: 0
NULL Forecast: 0
Negative final forecasts: 0

=== BEST ETS COUNT ===
Model
ETS_Seasonal    28
ETS_Level        6
Name: count, dtype: int64

=== BEST ETS BY MATERIAL ===


,MaterialKey,Model,MAE,RMSE,Bias
0,4,ETS_Level,973.000000,1207.040820,361.333160
1,5,ETS_Level,1473.666667,1821.372154,321.624988
2,6,ETS_Seasonal,258.128393,284.474018,106.598485
3,9,ETS_Seasonal,172.839914,195.183744,42.648064
4,11,ETS_Seasonal,1964.364296,2249.839821,1166.484297
5,12,ETS_Seasonal,982.081921,1124.744014,583.115181
6,13,ETS_Seasonal,691.335256,780.672761,170.596186
7,23,ETS_Seasonal,675.406758,763.429819,168.898374
8,24,ETS_Seasonal,675.406758,763.429819,168.898374
9,25,ETS_Seasonal,657.493070,743.220281,176.982601


Cell 6: Export kết quả

In [9]:
# ===== G09 - CELL 6: EXPORT ETS RESULTS =====

ets_predictions = ets_predictions.sort_values(
    ["MaterialKey", "Model", "ForecastMonth"]
).reset_index(drop=True)

ets_metrics = ets_metrics.sort_values(
    ["MaterialKey", "Model"]
).reset_index(drop=True)

ets_logs = ets_logs.sort_values(
    ["MaterialKey", "Model"]
).reset_index(drop=True)

best_ets = best_ets.sort_values(
    ["MaterialKey"]
).reset_index(drop=True)


ets_predictions.to_csv(
    "../output/G09_ets_predictions.csv",
    index=False
)

ets_metrics.to_csv(
    "../output/G09_ets_metrics.csv",
    index=False
)

ets_logs.to_csv(
    "../output/G09_ets_logs.csv",
    index=False
)

best_ets.to_csv(
    "../output/G09_best_ets_by_material.csv",
    index=False
)


print("=== G09 EXPORT COMPLETED ===")

print("Prediction rows:", len(ets_predictions))
print("Metric rows:", len(ets_metrics))
print("Log rows:", len(ets_logs))
print("Best ETS rows:", len(best_ets))

print(
    "Successful evaluations:",
    (ets_metrics["Status"] == "SUCCESS").sum()
)

print(
    "Fit failures:",
    (ets_metrics["Status"] == "FIT_FAILED").sum()
)

print(
    "Clipped forecasts:",
    ets_predictions["WasClipped"].sum()
)

=== G09 EXPORT COMPLETED ===
Prediction rows: 588
Metric rows: 105
Log rows: 105
Best ETS rows: 34
Successful evaluations: 98
Fit failures: 0
Clipped forecasts: 3


Setup + load cùng dataset

In [10]:
# ===== G10 - CELL 1: SETUP & LOAD DATA =====

import pandas as pd
import numpy as np
import warnings

from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

# Same locked dataset used by G06 and G09
df = pd.read_csv("../output/G06_forecast_dataset.csv")

df["Month"] = pd.to_datetime(df["Month"])

print("=== G10 DATA CHECK ===")
print("Rows:", len(df))
print("Materials:", df["MaterialKey"].nunique())
print("Min month:", df["Month"].min())
print("Max month:", df["Month"].max())

print("\nTrain rows:",
      (df["IsTrainPeriod"] == 1).sum())

print("Test rows:",
      (df["IsTestPeriod"] == 1).sum())

print("\nTier distribution:")
print(
    df[["MaterialKey", "DSSEligibilityTier"]]
    .drop_duplicates()["DSSEligibilityTier"]
    .value_counts()
)

=== G10 DATA CHECK ===
Rows: 1145
Materials: 35
Min month: 2022-07-01 00:00:00
Max month: 2025-05-01 00:00:00

Train rows: 941
Test rows: 204

Tier distribution:
DSSEligibilityTier
A    28
B     7
Name: count, dtype: int64
